In [ ]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Google GenAI SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U google-genai pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Official Google GenAI SDK
from google import genai
from google.genai import types
from google.genai.errors import APIError

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
# Never hardcode API keys directly in scripts!
# In Google Colab, use the Secrets Manager (🔑 icon on the left panel) as 'GEMINI_API_KEY'.
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini API Client initialized successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
✅ Google Gemini API Client initialized successfully!


In [ ]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (User, Model) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'model'             │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""

"\n1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:\n   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.\n   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.\n   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.\n\n2. THE STATELESSNESS MENTAL MODEL:\n   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.\n   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list\n     of previous (User, Model) turns and pass the cumulative array on every subsequent call.\n\n3. MESSAGE ROLES MAPPING:\n   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐\n   │ Role Type            │ OpenAI / Anthropic      │ Google Gemini API         │\n   ├──────────────────────┼─────────────────────────┼───────────────────────────┤\n   │ System Persona/Rules │ role: 'system'          │ config.system_instruction │\

In [ ]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION (UPDATED)
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.
"""
import numpy as np
def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "gemini-3.6-flash",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Calculates exact input tokens and estimates financial cost before calling the API."""
    # Count tokens using official Gemini Tokenizer
    token_resp = client.models.count_tokens(model=model_name, contents=text_prompt)
    input_tokens = token_resp.total_tokens

    # Official Rates per 1M tokens (USD)
    pricing = {
        "gemini-3.6-flash": {"in": 0.075, "out": 0.30},
        "gemini-1.5-pro":   {"in": 1.25,  "out": 5.00}
    }
    rate = pricing.get(model_name, pricing["gemini-3.6-flash"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": np.round(est_cost, 6),
        "cost_per_10k_calls": np.round(est_cost * 10000, 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
# Updated to gemini-3.6-flash
estimate = preflight_cost_estimate(sample_payload, model_name="gemini-3.6-flash")
print("=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"• {k:25s}: {v}")

=== 📊 PRE-FLIGHT TOKEN & COST AUDIT ===
• model                    : gemini-3.6-flash
• input_tokens             : 19
• estimated_output_tokens  : 500
• estimated_cost_usd       : 0.000151
• cost_per_10k_calls       : 1.51


In [ ]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429 / ResourceExhausted): Hit requests-per-minute (RPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Google Cloud infrastructure hiccup.
3. Network Timeouts: Connection dropped during streaming.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents "Thundering Herd" problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except APIError as e:
            if attempt == max_retries - 1:
                print(f"❌ Max retries reached. Fatal API Error: {e}")
                raise e
            # Calculate backoff delay with jitter
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.1, 0.8)
            print(f"⚠️ Warning: Transient API Error ({e.code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)

In [ ]:
# ==============================================================================
# SECTION 4: REUSABLE GEMINI WRAPPER & 3-TURN CHAT
# ==============================================================================
# Construct a reusable production function supporting:
# - Streaming responses (Low Time-To-First-Token)
# - System instructions
# - Dynamic temperature
# - Exponential backoff retry logic

def gemini_call(
    prompt,
    system_instruction=None,
    temperature=0.7,
    max_retries=3
):
    import time
    from google import genai
    from google.genai import types

    client = genai.Client()

    for attempt in range(max_retries):
        try:
            config = types.GenerateContentConfig(
                temperature=temperature,
                system_instruction=system_instruction
            )

            response = client.models.generate_content_stream(
                model="gemini-3.6-flash",
                contents=prompt,
                config=config
            )

            for chunk in response:
                if chunk.text:
                    print(chunk.text, end="")

            print()
            return

        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


    chat_history = []

    def send_chat_turn(user_message):
        chat_history.append(
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=user_message)]
            )
        )

        client = genai.Client()

        for attempt in range(3):
            try:
                response = client.models.generate_content_stream(
                    model="gemini-3.6-flash",
                    contents=chat_history,
                    config=types.GenerateContentConfig(
                        temperature=0.7,
                        system_instruction="You are a helpful SQL expert. Maintain context across the conversation."
                    )
                )

                response_text = ""

                for chunk in response:
                    if chunk.text:
                        print(chunk.text, end="")
                        response_text += chunk.text

                print()

                chat_history.append(
                    types.Content(
                        role="model",
                        parts=[types.Part.from_text(text=response_text)]
                    )
                )
                return

            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)


    send_chat_turn("What is the difference between a clustered and non-clustered index?")
    send_chat_turn("Which one is faster for range queries on primary keys?")
    send_chat_turn("Can a table have multiple of the faster one?")

In [ ]:
# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================
"""
🎓 STUDENT LAB ASSIGNMENT:
Build an end-to-end AI Application: "The Executive Resume Bullet & Impact Optimizer"

APPLICATION REQUIREMENTS:
1. Structured JSON Schema (Pydantic):
   - `original_bullet`: Raw user text
   - `xyz_formatted_bullet`: Rewritten using Google's XYZ Formula:
     "Accomplished [X], as measured by [Y], by doing [Z]"
   - `impact_metric`: The quantifiable numeric KPI
   - `action_verb`: Strong opening action verb
   - `seniority_score`: Integer rating (1 to 10) of executive presence
   - `critique`: 1-sentence explanation of what was improved
2. Interactive Revision History: Allow user to request a revision (multi-turn).
3. Streaming or Schema Parsing: Correctly parse and display output.
4. Error Handling: Enclose calls in retry blocks.
"""

# ==============================================================================
# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
# ==============================================================================

# TODO 1.1: Complete the Pydantic Schema

# ==============================================================================
# TASK 2: BUILD THE APPLICATION ENGINE
# ==============================================================================


    # TODO 2.1: Configure GenerateContentConfig with temperature=0.1, response_mime_type='application/json', and response_schema


    # TODO 2.2: Execute API call with exponential backoff


# ==============================================================================
# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
# ==============================================================================



'\n🎓 STUDENT LAB ASSIGNMENT:\nBuild an end-to-end AI Application: "The Executive Resume Bullet & Impact Optimizer"\n\nAPPLICATION REQUIREMENTS:\n1. Structured JSON Schema (Pydantic):\n   - `original_bullet`: Raw user text\n   - `xyz_formatted_bullet`: Rewritten using Google\'s XYZ Formula:\n     "Accomplished [X], as measured by [Y], by doing [Z]"\n   - `impact_metric`: The quantifiable numeric KPI\n   - `action_verb`: Strong opening action verb\n   - `seniority_score`: Integer rating (1 to 10) of executive presence\n   - `critique`: 1-sentence explanation of what was improved\n2. Interactive Revision History: Allow user to request a revision (multi-turn).\n3. Streaming or Schema Parsing: Correctly parse and display output.\n4. Error Handling: Enclose calls in retry blocks.\n'

In [22]:
from pydantic import BaseModel, Field
from typing import List
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

class ResumeOptimization(BaseModel):
    original_bullet: str
    optimized_bullet: str
    impact_score: float = Field(ge=0, le=10)
    action_verbs: List[str]
    skills_highlighted: List[str]
    ats_keywords: List[str]
    quantified_impact: str
    improvement_summary: str


def optimize_resume_bullet(
    original_bullet: str,
    target_role: str,
    industry: str,
    temperature: float = 0.4
) -> ResumeOptimization:

    system_instruction = """
    You are an expert executive resume strategist, recruiter, and ATS optimization specialist.
    Your task is to transform resume bullets into concise, achievement-oriented, high-impact statements.

    Follow these rules:
    1. Preserve the factual meaning of the original bullet.
    2. Never invent metrics, achievements, technologies, responsibilities, or outcomes.
    3. Begin the optimized bullet with a strong action verb.
    4. Emphasize measurable business or technical impact when evidence exists.
    5. Improve ATS keyword alignment with the target role and industry.
    6. Keep the optimized bullet concise and professional.
    7. Return only structured output matching the provided schema.
    """

    user_prompt = f"""
    Optimize this resume bullet.

    Original Bullet:
    {original_bullet}

    Target Role:
    {target_role}

    Industry:
    {industry}

    Analyze the bullet and provide:
    - The original bullet exactly as provided
    - An improved executive-level version
    - An impact score from 0 to 10
    - Strong action verbs used or recommended
    - Skills demonstrated
    - ATS keywords relevant to the target role
    - Quantified impact stated in the original bullet, or "Not provided" when absent
    - A concise explanation of the improvements
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=temperature,
                    response_mime_type="application/json",
                    response_schema=ResumeOptimization
                )
            )
            return ResumeOptimization.model_validate_json(response.text)

        except Exception:
            if attempt == 2:
                raise


original_bullet = "Worked on machine learning models for customer churn prediction."

target_role = "AI Engineer"
industry = "Technology"

result = optimize_resume_bullet(
    original_bullet=original_bullet,
    target_role=target_role,
    industry=industry
)

print(result.model_dump_json(indent=2))


{
  "original_bullet": "Worked on machine learning models for customer churn prediction.",
  "optimized_bullet": "Engineered predictive machine learning models to analyze customer churn and drive data-driven retention strategies.",
  "impact_score": 7.5,
  "action_verbs": [
    "Engineered",
    "Analyze",
    "Drive"
  ],
  "skills_highlighted": [
    "Machine Learning",
    "Predictive Modeling",
    "Customer Churn Analysis",
    "Data Science"
  ],
  "ats_keywords": [
    "Machine Learning",
    "Predictive Analytics",
    "Customer Churn",
    "AI Engineer",
    "Model Optimization"
  ],
  "quantified_impact": "Not provided",
  "improvement_summary": "Replaced weak passive phrasing ('Worked on') with a strong action verb ('Engineered'), emphasized predictive modeling capabilities, and aligned bullet structure with AI Engineer ATS keywords."
}


In [ ]:
# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GEMINI_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GEMINI_API_KEY")
"""

# Script to generate .gitignore locally in Colab
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("✅ '.gitignore' template created successfully!")